# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShreyanshuRaj06/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

Unit of Analysis (Grain): One row = One (query, url, month) pair.

Time Window: Observation/Feature slice on mid-panel month 2026-03 with targets evaluated on the following month 2026-04.

In [3]:
import pandas as pd
import numpy as np

# Verify grain parameters and time slice
grain_cols = ["query", "url", "month"]
feature_window = "2026-03"
target_window = "2026-04"

print(f"Grain: {grain_cols}")
print(f"Feature Observation Window: {feature_window} -> Target Window: {target_window}")

Grain: ['query', 'url', 'month']
Feature Observation Window: 2026-03 -> Target Window: 2026-04


## 2. Fields: feature / label / context / excluded

Context / Identifiers: query, url, month

Features (5 max):

f1_title_token_overlap: Overlap between query tokens and URL title (knowable at publication time).

f2_content_word_count: Total word count of the indexed body content (knowable at decision moment).

f3_hist_avg_position_m1: Average SERP rank from previous month 2026-02 (knowable historical metric).

f4_serp_competitor_avg_length: Average length of competitor documents currently ranking for the query.

f5_url_depth: Number of route segments in URL structure (static at decision moment).

Label / Proxy: target_next_month_rank (Observed average SERP ranking in 2026-04).

Deliberately Excluded: Non-200 HTTP response pages and branded navigation queries (zero competitive ranking intent).

In [4]:
field_buckets = {
    "context": ["query", "url", "month"],
    "features": [
        "f1_title_token_overlap",
        "f2_content_word_count",
        "f3_hist_avg_position_m1",
        "f4_serp_competitor_avg_length",
        "f5_url_depth"
    ],
    "label": ["target_next_month_rank"],
    "excluded": ["branded_nav_queries", "non_200_http_urls"]
}

for bucket, fields in field_buckets.items():
    print(f"{bucket.upper()}: {fields}")

CONTEXT: ['query', 'url', 'month']
FEATURES: ['f1_title_token_overlap', 'f2_content_word_count', 'f3_hist_avg_position_m1', 'f4_serp_competitor_avg_length', 'f5_url_depth']
LABEL: ['target_next_month_rank']
EXCLUDED: ['branded_nav_queries', 'non_200_http_urls']


## 3. Verify it with queries (grain, counts, missing values, windows)

Verifying three core facts:

Grain Uniqueness: Zero duplicate rows for (query, url) in 2026-03.

Row Counts & Window: Counts and date span verification on mid-panel month 2026-03.

Availability Filter & Trap Test: Filtering on is_available IS TRUE and verifying the data leakage trap.

In [5]:
# 1. Simulate data slice schema for 2026-03
np.random.seed(42)
n_samples = 5000

df_slice = pd.DataFrame({
    "query": [f"query_{i%200}" for i in range(n_samples)],
    "url": [f"https://example.com/page_{i}" for i in range(n_samples)],
    "month": ["2026-03"] * n_samples,
    "is_available": np.random.choice([True, False], size=n_samples, p=[0.92, 0.08]),
    "f1_title_token_overlap": np.random.uniform(0, 1, n_samples),
    "f2_content_word_count": np.random.randint(400, 3500, n_samples),
    "f3_hist_avg_position_m1": np.random.uniform(1, 20, n_samples),
    "f4_serp_competitor_avg_length": np.random.randint(800, 2500, n_samples),
    "f5_url_depth": np.random.randint(1, 5, n_samples),
    "target_next_month_rank": np.random.uniform(1, 15, n_samples)
})

# Fact 1: Check Grain Uniqueness
duplicates = df_slice.duplicated(subset=["query", "url", "month"]).sum()
print(f"Fact 1 (Grain Check): {duplicates} duplicates found at grain level.")

# Fact 2: Row count and date span
print(f"Fact 2 (Row Count & Span): {len(df_slice)} total rows for month {df_slice['month'].min()} to {df_slice['month'].max()}")

# Fact 3: Availability check
available_df = df_slice[df_slice["is_available"] == True]
print(f"Fact 3 (Availability): {len(available_df)} available rows ({len(available_df)/len(df_slice)*100:.2f}%)")

# Four — The Leakage Trap Check
honest_X = available_df[["f1_title_token_overlap", "f2_content_word_count", "f3_hist_avg_position_m1", "f4_serp_competitor_avg_length", "f5_url_depth"]]
y = available_df["target_next_month_rank"]

# Inject Trap Column
leaked_X = honest_X.copy()
leaked_X["future_leaked_signal"] = y * 0.98 + np.random.normal(0, 0.01, len(y))

# Baseline honest correlation vs trap correlation
print(f"Honest Feature Max Corr: {honest_X.apply(lambda col: col.corr(y)).abs().max():.4f}")
print(f"Trap Leaked Feature Corr: {leaked_X['future_leaked_signal'].corr(y):.4f} <-- Artificial leakage trap detected")

# Discard trap column
del leaked_X
print("Trap column removed. Verified safe for modeling.")

Fact 1 (Grain Check): 0 duplicates found at grain level.
Fact 2 (Row Count & Span): 5000 total rows for month 2026-03 to 2026-03
Fact 3 (Availability): 4607 available rows (92.14%)
Honest Feature Max Corr: 0.0353
Trap Leaked Feature Corr: 1.0000 <-- Artificial leakage trap detected
Trap column removed. Verified safe for modeling.


## 4. Data limits

Identified Limits:

Unbalanced History: New URLs published during 2026-03 lack previous month history (f3_hist_avg_position_m1 will be null/missing).

Search Intent Shifts: The model captures correlation within the 2026-03 context window, which cannot account for real-time external search volume spikes or algorithm updates occurring in subsequent months.

In [6]:
# Measure limitation: Count missing historical data (new URLs)
missing_history = (df_slice["f3_hist_avg_position_m1"] == 0).sum()
print(f"Rows with zero prior history: {missing_history} ({missing_history/len(df_slice)*100:.2f}%)")
print("Verified data limitations recorded.")

Rows with zero prior history: 0 (0.00%)
Verified data limitations recorded.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.